# 3DEP Extras — Terrain Feature Factory

Build reusable terrain features from the USGS 3DEP 10m DEM for any AOI.

In [3]:
import ee
import numpy as np
import pandas as pd

ee.Authenticate()
ee.Initialize(project="shrubwise-dc-488219")

THREEDEP_10M = ee.ImageCollection("USGS/3DEP/10m_collection")
AOI_BLISS = ee.Geometry.Rectangle([-120.1018846, 38.99274873, -120.0899834, 39.0020357], geodesic=False)


In [4]:
def threedep_mosaic():
    return THREEDEP_10M.mosaic().select("elevation").toFloat().rename("elevation")

def terrain_feature_cube(*, smooth=False):
    dem = threedep_mosaic()
    if smooth:
        dem = dem.focal_mean(radius=1, units="pixels").rename("elevation")

    slope = ee.Terrain.slope(dem).rename("slope_deg")
    aspect = ee.Terrain.aspect(dem).rename("aspect_deg")

    aspect_rad = aspect.multiply(np.pi / 180.0)
    northness = aspect_rad.cos().rename("northness")
    eastness = aspect_rad.sin().rename("eastness")

    relief_3 = dem.focal_max(radius=1, units="pixels").subtract(
        dem.focal_min(radius=1, units="pixels")
    ).rename("relief_3px")
    
    relief_7 = dem.focal_max(radius=3, units="pixels").subtract(
        dem.focal_min(radius=3, units="pixels")
    ).rename("relief_7px")
    
    relief_15 = dem.focal_max(radius=7, units="pixels").subtract(
        dem.focal_min(radius=7, units="pixels")
    ).rename("relief_15px")

    return ee.Image.cat([dem, slope, aspect, northness, eastness, relief_3, relief_7, relief_15])

cube = terrain_feature_cube(smooth=False)
print(cube.bandNames().getInfo())


['elevation', 'slope_deg', 'aspect_deg', 'northness', 'eastness', 'relief_3px', 'relief_7px', 'relief_15px']


In [5]:
def summarize_feature_cube(image, aois, *, scale=10):
    reducer = (ee.Reducer.count()
               .combine(ee.Reducer.mean(), sharedInputs=True)
               .combine(ee.Reducer.median(), sharedInputs=True)
               .combine(ee.Reducer.min(), sharedInputs=True)
               .combine(ee.Reducer.max(), sharedInputs=True))
    rows = []
    for name, geom in aois.items():
        stats = image.reduceRegion(reducer, geom, scale, maxPixels=1e9, bestEffort=True).getInfo()
        row = {"aoi": name, **stats}
        rows.append(row)
    return pd.DataFrame(rows).set_index("aoi")

AOIS = {"DL_Bliss": AOI_BLISS}
summarize_feature_cube(cube, AOIS)


,aspect_deg_count,aspect_deg_max,aspect_deg_mean,aspect_deg_median,aspect_deg_min,eastness_count,eastness_max,eastness_mean,eastness_median,eastness_min,...,relief_7px_count,relief_7px_max,relief_7px_mean,relief_7px_median,relief_7px_min,slope_deg_count,slope_deg_max,slope_deg_mean,slope_deg_median,slope_deg_min
aoi,,,,,,,,,,,,,,,,,,,,,
DL_Bliss,13936,260.594513,237.507435,231.039108,231.039108,13936,-0.777575,-0.823312,-0.777575,-0.986557,...,13936,60.621094,8.232701,0.00136,0,13936,0.789642,0.744548,0.789642,0.583597
